In [8]:
import numpy as np
from scipy.stats import norm
from scipy.linalg import solve_banded

In [9]:
# =============================================================================
# Black-Scholes European Put (closed form)
# =============================================================================
 
def bs_european_put(S, K, r, sigma, T):
    """Black-Scholes price for a European put."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

In [10]:
# =============================================================================
# Finite Difference Solver for American Put
# =============================================================================
def fd_american_put(S0, K, r, sigma, T,
                    N_S=1000,           # stock-price steps
                    N_t_per_yr=40_000): # time steps per year
    """
    Fully-implicit finite difference scheme for the American put.
    Solves the tridiagonal system at each time step via scipy's
    banded solver, then applies the early-exercise constraint.
 
    Returns the option price at S0.
    """
    N_t  = int(N_t_per_yr * T)
    Smax = 4.0 * K
    dS   = Smax / N_S
    dt   = T / N_t
 
    S = np.linspace(0, Smax, N_S + 1)           # shape (N_S+1,)
 
    # Terminal payoff
    V = np.maximum(K - S, 0.0)
 
    # Interior nodes: j = 1 … N_S-1
    j   = np.arange(1, N_S)                      # shape (N_S-1,)
    Sj  = j * dS
 
    alpha = 0.5 * dt * sigma**2 * j**2           # ½σ²j²Δt
    beta  = 0.5 * dt * r * j                     # ½rjΔt
 
    lower = -(alpha - beta)                       # a_j
    diag  =  1.0 + 2*alpha + r*dt                # b_j
    upper = -(alpha + beta)                       # c_j
 
    # scipy solve_banded expects (ab, b) where ab has shape (3, N)
    # Row 0: upper diagonal (shifted right by 1)
    # Row 1: main diagonal
    # Row 2: lower diagonal (shifted left by 1)
    ab = np.zeros((3, N_S - 1))
    ab[0, 1:]  = upper[:-1]    # upper diagonal
    ab[1, :]   = diag           # main diagonal
    ab[2, :-1] = lower[1:]      # lower diagonal
 
    # Boundary conditions: V[0] = K, V[N_S] = 0  (deep ITM / deep OTM)
    V[0]    = K
    V[N_S]  = 0.0
 
    intrinsic = np.maximum(K - S, 0.0)
 
    for _ in range(N_t):
        rhs = V[1:N_S].copy()
        # Absorb boundary values into rhs
        rhs[0]   -= lower[0]  * V[0]
        rhs[-1]  -= upper[-1] * V[N_S]
 
        V_new = solve_banded((1, 1), ab, rhs)
 
        # Early-exercise constraint
        V[1:N_S] = np.maximum(V_new, intrinsic[1:N_S])
 
    # Interpolate to S0
    idx  = S0 / dS
    lo   = int(idx)
    frac = idx - lo
    lo   = np.clip(lo, 0, N_S)
    hi   = np.clip(lo + 1, 0, N_S)
    price = V[lo] * (1 - frac) + V[hi] * frac
    return float(price)
 

In [ ]:
# =============================================================================
# Laguerre basis functions  (weighted, as in the paper)
#
#   L0(x) = exp(-x/2)
#   L1(x) = exp(-x/2) * (1 - x)
#   L2(x) = exp(-x/2) * (1 - 2x + x^2/2)
#
#   Applied to the *normalised* stock price  x = S / K
# =============================================================================
 
def laguerre_basis(S, K):
    """
    Returns design matrix  X  of shape (n_paths, 4):
      [1,  L0(S/K),  L1(S/K),  L2(S/K)]
 
    Numerically safe: clips S/K to [1e-8, 100] before evaluating
    the weighted polynomials, preventing exp/quadratic overflow.
    """
    x = np.clip(S / K, 1e-4, 100.0)     # guard against extreme values
    e = np.exp(-x / 2)                   # always in (0, 1] after clipping
    L0 = e
    L1 = e * (1.0 - x)
    L2 = e * (1.0 - 2.0*x + 0.5*x**2)
    return np.column_stack([np.ones_like(x), L0, L1, L2])



# =============================================================================
# LSM Algorithm
# =============================================================================
 
def lsm_american_put(S0, K, r, sigma, T, simulated_paths,
                     n_steps_per_year=50,
                     n_paths=100_000):
    """
    Price an American put using the Longstaff-Schwartz (2001) LSM method.
 
    Parameters
    ----------
    S0            : float  — current stock price
    K             : float  — strike price
    r             : float  — risk-free rate (continuous, annualised)
    sigma         : float  — volatility of returns (annualised)
    T             : float  — time to expiry in years
    n_steps_per_year : int — number of exercise opportunities per year
    n_paths       : int    — total simulation paths (must be even)
 
    Returns
    -------
    american_price : float
    european_price : float  (same paths, European payoff)
    std_error      : float  (std error of the American price estimate)
    """
    n_steps = int(T * n_steps_per_year)
    dt      = T / n_steps
    disc    = np.exp(-r * dt)             # one-step discount factor
 
    # ── Simulate paths ───────────────────────────────────────────────────────
    S = simulate_paths(S0, r, sigma, T, n_steps, n_paths)
    # S shape: (n_paths, n_steps + 1)
 
    # ── Initialise cash-flow matrix with terminal payoffs ────────────────────
    #   cashflow[i] = cash flow along path i (amount and *when* it occurs
    #                 is tracked implicitly — we discount as we roll back)
    cashflow = np.maximum(K - S[:, -1], 0.0)   # payoff at expiry
 
    # ── Backward recursion ───────────────────────────────────────────────────
    for t in range(n_steps - 1, 0, -1):        # t = n_steps-1, ..., 1
 
        # Discount cashflows one step forward (they are currently priced at t+1)
        cashflow *= disc
 
        S_t = S[:, t]
        intrinsic = np.maximum(K - S_t, 0.0)
 
        # Only use in-the-money paths for the regression
        itm = intrinsic > 0
        if itm.sum() == 0:
            continue
 
        # Build basis matrix for ITM paths
        X = laguerre_basis(S_t[itm], K)        # shape: (n_itm, 4)
        Y = cashflow[itm]                       # continuation values (discounted)
 
        # ── Robust OLS with ridge fallback ────────────────────────────────────
        # np.linalg.lstsq is numerically stable via SVD, but if the design
        # matrix is still nearly singular (e.g. very few ITM paths, or all
        # paths at the same stock price) the fitted values can be NaN/inf.
        # We add a tiny ridge (Tikhonov) penalty as a fallback: it changes
        # prices by < 1e-6 in normal operation but prevents blow-up.
        try:
            coeffs = np.linalg.lstsq(X, Y, rcond=None)[0]
            continuation_hat = X @ coeffs
            # Detect blow-up
            if not np.all(np.isfinite(continuation_hat)):
                raise np.linalg.LinAlgError("non-finite fitted values")
        except np.linalg.LinAlgError:
            # Ridge regression fallback: (X'X + λI)^{-1} X'Y
            lam = 1e-6 * (X.T @ X).diagonal().mean()
            XtX = X.T @ X + lam * np.eye(X.shape[1])
            coeffs = np.linalg.solve(XtX, X.T @ Y)
            continuation_hat = X @ coeffs
 
        # Final safety net: replace any residual NaN/inf with 0
        # (treat as "no continuation value" → forces exercise if ITM)
        continuation_hat = np.where(np.isfinite(continuation_hat),
                                    continuation_hat, 0.0)
 
        # Exercise decision for ITM paths
        exercise_now = intrinsic[itm] >= continuation_hat
 
        # Update cashflows: if exercising now, replace with intrinsic
        cashflow[itm] = np.where(exercise_now, intrinsic[itm], cashflow[itm])
 
    # ── Discount back to t = 0 and average ───────────────────────────────────
    # cashflow currently represents the value at t = dt (one period from 0)
    cashflow_t0 = cashflow * disc               # discount to t = 0
 
    american_price = cashflow_t0.mean()
    std_error       = cashflow_t0.std() / np.sqrt(n_paths)
 
    # European put (same paths, same discount)
    european_payoff = np.maximum(K - S[:, -1], 0.0) * np.exp(-r * T)
    european_price  = european_payoff.mean()
 
    return american_price, european_price, std_error


SyntaxError: invalid syntax (2780392530.py, line 33)